# CCO11: Visium HD

In [1]:
import scanpy as sc
import squidpy as sq
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pysam
from Bio import SeqIO
from pathlib import Path
import anndata as ad

In [2]:
WORK_DIR = "/home/herbert/PyProjects/cocultured_organ/workspace/CCO11"
DATA_DIR = "/home/herbert/PyProjects/cocultured_organ/data/260416"
os.system(f"mkdir -p {WORK_DIR}")

0

In [3]:
os.system(f"mkdir -p {WORK_DIR}/data/orig")

0

# -2. rerun SpaceRanger

In [4]:
os.system(f"mkdir -p {WORK_DIR}/spaceranger")

0

In [6]:
os.system(f"mkdir -p {WORK_DIR}/spaceranger/reference")

0

In [10]:
os.system(f"qsub {WORK_DIR}/spaceranger/run/run.sh")

requested Hard Resources
  memory (s_vmem): 4G = The job requires 4GB of memory per slot.
  slots (def_slot): 32 = The job requires 3200% of CPU.
  total memory: 128G = The job requires 128GB of memory.
Your job 121702474 ("run.sh") has been submitted


0

# -1. re-map

In [12]:
os.system(f"mkdir -p {WORK_DIR}/remap")
os.system(f"mkdir -p {WORK_DIR}/remap/ref")

0

In [42]:
os.system(f"qsub {WORK_DIR}/remap/ref/build_star_index.sh")

requested Hard Resources
  memory (s_vmem): 4G = The job requires 4GB of memory per slot.
  slots (def_slot): 32 = The job requires 3200% of CPU.
  total memory: 128G = The job requires 128GB of memory.
Your job 121721249 ("build_star_index.sh") has been submitted


0

In [43]:
os.system(f"qsub {WORK_DIR}/remap/map.sh")

requested Hard Resources
  memory (s_vmem): 4G = The job requires 4GB of memory per slot.
  slots (def_slot): 32 = The job requires 3200% of CPU.
  total memory: 128G = The job requires 128GB of memory.
Your job 121721857 ("map.sh") has been submitted


0

In [34]:
with open(f"{WORK_DIR}/remap/ref/Homo_sapiens.GRCh38.dna.temp.fa", "w") as f:
    for record in SeqIO.parse(f"{WORK_DIR}/remap/ref/Homo_sapiens.GRCh38.dna.toplevel.fa", "fasta"):
        header = None
        try:
            header = f"GRCh38_ch{int(record.id)}"
        except:
            if record.id in ("X", "Y", "MT"):
                header = f"GRCh38_ch{record.id}"
        if header is None:
            continue
        f.write(f">{header}\n")
        f.write(f"{str(record.seq)}\n")

with open(f"{WORK_DIR}/remap/ref/Rattus_norvegicus.mRatBN7.2.dna.temp.fa", "w") as f:
    for record in SeqIO.parse(f"{WORK_DIR}/remap/ref/Rattus_norvegicus.mRatBN7.2.dna.toplevel.fa", "fasta"):
        header = None
        try:
            header = f"mRatBN7_ch{int(record.id)}"
        except:
            if record.id in ("X", "Y", "MT"):
                header = f"mRatBN7_ch{record.id}"
        if header is None:
            continue
        f.write(f">{header}\n")
        f.write(f"{str(record.seq)}\n")

In [41]:
def rewrite_gtf_chroms(in_gtf, out_gtf, prefix):
    in_gtf = Path(in_gtf)
    out_gtf = Path(out_gtf)

    with in_gtf.open() as fin, out_gtf.open("w") as fout:
        for line in fin:
            if line.startswith("#"):
                fout.write(line)
                continue

            fields = line.rstrip("\n").split("\t")
            chrom = fields[0]

            fields[0] = f"{prefix}_ch{chrom}"

            fout.write("\t".join(fields) + "\n")


rewrite_gtf_chroms(
    f"{WORK_DIR}/remap/ref/Homo_sapiens.GRCh38.filtered.gtf",
    f"{WORK_DIR}/remap/ref/Homo_sapiens.GRCh38.renamed.gtf",
    "GRCh38"
)

rewrite_gtf_chroms(
    f"{WORK_DIR}/remap/ref/Rattus_norvegicus.mRatBN7.2.filtered.gtf",
    f"{WORK_DIR}/remap/ref/Rattus_norvegicus.mRatBN7.2.renamed.gtf",
    "mRatBN7"
)

In [44]:
samfile = pysam.AlignmentFile(f"{WORK_DIR}/remap/graftho_R2_Aligned.sortedByCoord.out.bam", "rb")
# samfile.check_index()

In [45]:
for i, s in enumerate(samfile):
    if i > 10:
        break
    print(s)

VH00387:807:AAHNWG2M5:1:1101:37962:32540	0	#0	17007	3	7S42M1S	*	0	0	AACATGGTCTCAGGCACCTGGCCCAGGTCTGGCACATAGAAGTAGTTCTA	array('B', [40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40])	[('NH', 2), ('HI', 1), ('AS', 41), ('nM', 0)]
VH00387:807:AAHNWG2M5:1:1210:67615:23567	0	#0	17007	3	7S42M1S	*	0	0	AACATGGTCTCAGGCACCTGGCCCAGGTCTGGCACATAGAAGTAGTTCTA	array('B', [40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 24, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40])	[('NH', 2), ('HI', 1), ('AS', 41), ('nM', 0)]
VH00387:807:AAHNWG2M5:1:1210:71175:26444	0	#0	17007	3	7S42M1S	*	0	0	AACATGGTCTCAGGCACCTGGCCCAGGTCTGGCACATAGAAGTAGTTCTA	array('B', [40, 40, 40, 40, 40, 40, 40, 40, 24, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 24, 40, 40, 40, 40, 2

In [46]:
for i, read in enumerate(samfile.fetch(until_eof=True)):
    if i == 20:
        break
    print(f"""
{read.query_name}
strand: {"-" if read.is_reverse else "+"}
chrom: {read.reference_name}
chrom_id: {read.reference_id}
start: {read.reference_start}
end: {read.reference_end}
cigar: {read.cigarstring}
splice: {read.cigartuples}
tags: {dict(read.tags)}
""")


VH00387:807:AAHNWG2M5:1:2311:44930:7172
strand: +
chrom: GRCh38_ch1
chrom_id: 0
start: 17006
end: 17048
cigar: 7S42M1S
splice: [(4, 7), (0, 42), (4, 1)]
tags: {'NH': 2, 'HI': 1, 'AS': 41, 'nM': 0}


VH00387:807:AAHNWG2M5:1:2311:45252:7304
strand: +
chrom: GRCh38_ch1
chrom_id: 0
start: 17006
end: 17048
cigar: 7S42M1S
splice: [(4, 7), (0, 42), (4, 1)]
tags: {'NH': 2, 'HI': 1, 'AS': 41, 'nM': 0}


VH00387:807:AAHNWG2M5:1:2407:57731:18190
strand: +
chrom: GRCh38_ch1
chrom_id: 0
start: 17006
end: 17048
cigar: 7S42M1S
splice: [(4, 7), (0, 42), (4, 1)]
tags: {'NH': 2, 'HI': 1, 'AS': 41, 'nM': 0}


VH00387:807:AAHNWG2M5:1:1101:37962:32540
strand: +
chrom: GRCh38_ch1
chrom_id: 0
start: 187528
end: 187570
cigar: 7S42M1S
splice: [(4, 7), (0, 42), (4, 1)]
tags: {'NH': 2, 'HI': 2, 'AS': 41, 'nM': 0}


VH00387:807:AAHNWG2M5:1:1210:67615:23567
strand: +
chrom: GRCh38_ch1
chrom_id: 0
start: 187528
end: 187570
cigar: 7S42M1S
splice: [(4, 7), (0, 42), (4, 1)]
tags: {'NH': 2, 'HI': 2, 'AS': 41, 'nM': 0}

# 0. convert to h5ad

In [20]:
for group in ("S11__graftho-JW-030626", "S11__graftvasho-JW-032626"):
    for bin_size in ("002um", "008um", "016um"):
        df = pd.read_parquet(f"{DATA_DIR}/{group}/outs/binned_outputs/square_{bin_size}/spatial/tissue_positions.parquet")
        df.to_csv(f"{DATA_DIR}/{group}/outs/binned_outputs/square_{bin_size}/spatial/tissue_positions_list.csv", index=False, header=False)
        os.system(f"ln -sfn {DATA_DIR}/{group}/outs/spatial/* {DATA_DIR}/{group}/outs/binned_outputs/square_{bin_size}/spatial")

In [22]:
# for group_path, group in zip(("S11__graftho-JW-030626", "S11__graftvasho-JW-032626"), ("HO", "VasHO")):
#     for res in ("002um", "008um", "016um"):
#         adata = sc.read_visium(
#             path=f"/home/herbert/PyProjects/cocultured_organ/data/260416/spaceranger/{group_path}/outs/binned_outputs/square_{res}", 
#             count_file="filtered_feature_bc_matrix.h5",
#         )
#         adata.var_names_make_unique()
#         adata.write(f"{WORK_DIR}/data/orig/{group}.{res}.h5ad")

# 1. process data

In [15]:
os.system(f"mkdir -p {WORK_DIR}/data/processed")
os.system(f"mkdir -p {WORK_DIR}/data/clustered")
os.system(f"mkdir -p {WORK_DIR}/umap")
os.system(f"mkdir -p {WORK_DIR}/dotplot")
os.system(f"mkdir -p {WORK_DIR}/spatial")

0

In [69]:
res = "008um"
# for group, group_long, cluster_index in zip(("HO", ), ("S11__graftho-JW-030626", ), (6, )):
for group, group_long, cluster_index in zip(("HO", "VasHO"), ("S11__graftho-JW-030626", "S11__graftvasho-JW-032626"), (6, 5)):
    adata = sc.read_h5ad(f"{WORK_DIR}/data/orig/{group}.{res}.h5ad")
    df = pd.read_csv(f"{DATA_DIR}/spaceranger/{group_long}/outs/binned_outputs/square_{res}/analysis/clustering/gene_expression_graphclust/clusters.csv", index_col=0)
    
    df_sub = df[df["Cluster"] == cluster_index]
    adata_sub = adata[df_sub.index]

    sc.pp.normalize_total(adata_sub, target_sum=1e4)
    sc.pp.log1p(adata_sub)
    adata_sub.write(f"{WORK_DIR}/data/processed/{group}.{res}.h5ad")
    # sc.tl.pca(adata_sub, svd_solver="arpack")
    # sc.pp.neighbors(adata_sub, n_neighbors=15, n_pcs=30)
    # sc.tl.leiden(adata_sub, resolution=0.6)
    # sc.tl.umap(adata_sub)
    
    # # UMAP cluster
    # sc.pl.umap(adata_sub, color="leiden", title=group)
    # sc.pl.dotplot(adata_sub, marker_dict, groupby="leiden", title=group)

/home/herbert/anaconda3/envs/cco/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:169: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
/home/herbert/anaconda3/envs/cco/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:169: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [5]:
marker_dict = [
    "CDH5", "PECAM1", "CLDN5", "KDR", "NOS3", "VWF", "CD34", "ENG", "TEK", "PLVAP",
    "VEGFA", "PDPN", "MYH7", "ACTC1", "TNNT2", "TNNI1", "TNNI3", "MYL2", "MYL7",
    "ACTA2", "TAGLN", "MYH11", "CNN1", "NOTCH3", "PDGFRB", "RGS5", "CSPG4", "MGP", "ABCC9",
    "NR2F1", "CXCL12", "PDGFRA", "C7", "EDN1", "COL1A1", "COL1A2", "FN1", "ITGA11", "LGALS1",
    "FBLN2", "SERPINE2", "THY1", "KRT8", "CDH1", "EPCAM"
]

adata_ho = sc.read_h5ad(f"{WORK_DIR}/data/processed/HO.008um.h5ad")
adata_vho = sc.read_h5ad(f"{WORK_DIR}/data/processed/VasHO.008um.h5ad")

adata_all = ad.concat(
    [adata_ho, adata_vho], 
    axis=0, 
    join="inner", 
    label="batch",  
    keys=["HO", "VasHO"]
)

sc.tl.pca(adata_all, svd_solver="arpack")
sc.pp.neighbors(adata_all, n_neighbors=15, n_pcs=30)
sc.tl.leiden(adata_all, resolution=0.5)
sc.tl.umap(adata_all)

# UMAP cluster
ax = sc.pl.umap(adata_all, color="leiden", show=False)
plt.savefig(f"{WORK_DIR}/umap/all.png", bbox_inches="tight")
plt.savefig(f"{WORK_DIR}/umap/all.svg", bbox_inches="tight")
plt.close()

ax = sc.pl.dotplot(adata_all, marker_dict, groupby="leiden", standard_scale="var", dot_max=0.6, show=False)
plt.savefig(f"{WORK_DIR}/dotplot/all.png", bbox_inches="tight")
plt.savefig(f"{WORK_DIR}/dotplot/all.svg", bbox_inches="tight")
plt.close()

adata_all.write(f"{WORK_DIR}/data/clustered/all.h5ad")
for group in ("HO", "VasHO"):
    adata = adata_all[adata_all.obs["batch"] == group]
    adata.write(f"{WORK_DIR}/data/clustered/{group}.h5ad")

/home/herbert/anaconda3/envs/cco/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/herbert/anaconda3/envs/cco/lib/python3.8/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot_ax.scatter(x, y, **kwds)


In [40]:
def plot_spatial(adata, adata_ori, figsize=(8,8), s=1, xlim=(-500, 0), ylim=(-500, 0), title="Spatial", s_legend=5, color_label=None, color_gene=None, color_gene_range=(0, 5), color_label_dict=None):
    common = adata.obs_names.intersection(adata_ori.obs_names)
    adata_sub = adata[common].copy()
    adata_ori_sub = adata_ori[common].copy()
    adata_sub.obsm["spatial"] = adata_ori_sub.obsm["spatial"].copy()
    adata_sub.uns["spatial"] = adata_ori.uns["spatial"].copy()
    
    fig, ax = plt.subplots(figsize=figsize)
    
    if color_label != None:
        # for l in adata_sub.obs[color_label].unique():
        all_labels = list(adata_sub.obs[color_label].unique())
        all_labels.sort()
        for l in all_labels:
            idx = adata_sub.obs[color_label] == l
            if color_label_dict is None:
                ax.scatter(
                    -adata_sub.obs.loc[idx, "array_col"],
                    -adata_sub.obs.loc[idx, "array_row"],
                    s=s,
                    label=l
                )
            else:
                ax.scatter(
                    -adata_sub.obs.loc[idx, "array_col"],
                    -adata_sub.obs.loc[idx, "array_row"],
                    s=s,
                    label=l, 
                    color=color_label_dict[l]
                )
        ax.legend(markerscale=s_legend, bbox_to_anchor=(1.05, 1), loc="upper left")
        
    if color_gene != None:
        scatter = ax.scatter(
            -adata_sub.obs.loc[:, "array_col"],
            -adata_sub.obs.loc[:, "array_row"],
            s=s,
            c=adata_sub[:,color_gene].X.toarray().ravel(),
            cmap='Reds', 
            vmin=color_gene_range[0], 
            vmax=color_gene_range[1],
        )
        cbar = plt.colorbar(scatter, ax=ax)
        
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(title)
    return fig, ax

In [28]:
celltype_marker_genes = {
    "EC": ['CHD5', 'PECAM1'], 
    "CM": ['TNNI1', 'ACTN2', 'MYH7', 'MYL2',], 
    "PVSC": ['PDGFRB', 'ACTA2', 'MYH11', 'COL1A1']
}

colormap_cell_types = {
    "CM": (60/255, 170/255, 60/255),
    "EC": (230/255, 80/255, 160/255),
    "EP": (141/255, 160/255, 203/255),
    "PVSC": (255/255, 210/255, 60/255),
    "UN": (190/255, 190/255, 190/255),
}

In [41]:
for group in ("HO", "VasHO"):
    adata = sc.read_h5ad(f"{WORK_DIR}/data/clustered/{group}.h5ad")
    adata_ori = sc.read_h5ad(f"{WORK_DIR}/data/orig/{group}.008um.h5ad")
    
    adata.obs_names = adata.obs_names.str.replace(r"-1$", "", regex=True)
    adata_ori.obs_names = adata_ori.obs_names.str.replace(r"-1$", "", regex=True)
    
    common = adata.obs_names.intersection(adata_ori.obs_names)
    adata = adata[common].copy()
    adata_ori_sub = adata_ori[common].copy()
    adata.obsm["spatial"] = adata_ori_sub.obsm["spatial"].copy()
    adata.uns["spatial"] = adata_ori.uns["spatial"].copy()
    assert (adata.obs_names == adata_ori_sub.obs_names).all()
    
    if group == "VasHO":
        s = 4
        xlim = (-360, -220)
        ylim = (-470, -330)
    else:
        s = 0.5
        xlim = (-500, -150)
        ylim = (-500, -150)
        
    fig, ax = plot_spatial(adata=adata, adata_ori=adata_ori, s=s, xlim=xlim, ylim=ylim, color_label="leiden", title=f"Spatial: {group}")
    plt.savefig(f"{WORK_DIR}/spatial/{group}.leiden.png", bbox_inches="tight")
    plt.savefig(f"{WORK_DIR}/spatial/{group}.leiden.svg", bbox_inches="tight")
    plt.close()

    marker_genes = ("PECAM1", "CDH5", "KDR")
    for marker_gene in marker_genes:
        fig, ax = plot_spatial(adata=adata, adata_ori=adata_ori, s=s, xlim=xlim, ylim=ylim, color_gene=marker_gene, title=f"Spatial: {marker_gene}")
        plt.savefig(f"{WORK_DIR}/spatial/{group}.{marker_gene}.png", bbox_inches="tight")
        plt.savefig(f"{WORK_DIR}/spatial/{group}.{marker_gene}.svg", bbox_inches="tight")
        plt.close()
        # plt.show()

    ax = sc.pl.umap(adata, color="leiden", title=f"UMAP: {group}", show=False)
    plt.savefig(f"{WORK_DIR}/umap/{group}.png", bbox_inches="tight")
    plt.savefig(f"{WORK_DIR}/umap/{group}.svg", bbox_inches="tight")
    plt.close()

    for ct in celltype_marker_genes:
        genes_present = [g for g in celltype_marker_genes[ct] if g in adata.var_names]
        X = adata[:, genes_present].X
        adata.obs[ct] = np.asarray((X > 0).sum(axis=1) > 0).ravel()
    adata.obs["celltype"] = np.select([
        adata.obs["EC"],
        adata.obs["PVSC"],
        adata.obs["CM"], ],["EC", "PVSC", "CM", ],
        default="UN"
    )
    
    fig, ax = plot_spatial(
        adata=adata, 
        adata_ori=adata_ori, 
        s=s, 
        xlim=xlim, 
        ylim=ylim, 
        color_label="celltype", 
        title=f"Spatial: {group}", 
        color_label_dict=colormap_cell_types
    )
    plt.savefig(f"{WORK_DIR}/spatial/{group}.celltype.png", bbox_inches="tight")
    plt.savefig(f"{WORK_DIR}/spatial/{group}.celltype.svg", bbox_inches="tight")
    plt.close()

/home/herbert/anaconda3/envs/cco/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/herbert/anaconda3/envs/cco/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


In [27]:
# for ct in celltype_marker_genes:
#     genes_present = [g for g in celltype_marker_genes[ct] if g in adata.var_names]
#     X = adata[:, genes_present].X
#     adata.obs[ct] = np.asarray((X > 0).sum(axis=1) > 0).ravel()

# adata.obs["celltype"] = np.select([
#     adata.obs["EC"],
#     adata.obs["PVSC"],
#     adata.obs["CM"],
# ],["EC", "PVSC", "CM", ],
#     default="UN"
# )

# fig, ax = plot_spatial(
#     adata=adata, 
#     adata_ori=adata_ori, 
#     s=s, 
#     xlim=xlim, 
#     ylim=ylim, 
#     color_label="celltype", 
#     title=f"Spatial: {group}", 
#     color_label_dict=colormap_cell_types
# )